# DATA 298A Team 6 — Workbook 2 (report sections 3 and 4)

**Neurosymbolic verification of LLM-generated clinical discharge and treatment plans.** Team 6, Section 22, Topic 12.

This notebook produces every table, sample and figure Workbook 2 asks for, in rubric order:

| Rubric item | Section here | What it produces |
|---|---|---|
| 3.1 Data process | 3.1 | process table and diagram |
| 3.2 Data collection | 3.2 | sources, parameters, quantity, checksum check, raw samples |
| 3.3 Data pre-processing | 3.3 | duplicates, missing values, validation (passes on clean data, fails on a corrupted copy), cleaned samples |
| 3.4 Data transformation | 3.4 | transformation log, parsed labs, case table, model input format, ML feature matrix |
| 3.5 Data preparation | 3.5 | frozen train / validation / test splits, leakage checks, prepared samples, exported datasets |
| 3.6 Data statistics | 3.6 | stage summary (raw → pre-processed → transformed → prepared) and six figures |
| 4.1 Model proposals | 4.1 | five models plus the reproduced baseline |
| 4.2 Model supports | 4.2 | measured environment, platform architecture and ML data-flow diagrams |
| 4.3 Comparison and justification | 4.3 | comparison table, justification per model |
| 4.4 Evaluation methods | 4.4 | metrics per model, **run** on the validation split: majority class, Model E (TF-IDF + logistic regression), Model A (local LLM on the GPU) |
| 4.5 Data pipeline demo | 4.5 | DuckDB warehouse (raw → staging → mart), data-quality gate, run log, monitoring, scheduling |

Figures and aggregate tables are written to `results/workbook2/` (gitignored) so they can be pasted into `T6.A6.docx`.

## Before you run

1. **Python environment** (repository root): `uv venv --python 3.13 .venv && uv pip install --python .venv/bin/python -r requirements.txt jupyter` (Python 3.13: pandas 2.2.3 has no wheel for 3.14), then pick the `.venv` kernel
2. **Data**: your own PhysioNet copy in `MIMIC_DIR` (default `~/data/mimic`), laid out as `mimic-iv-ext-cdm-1.1/` and `physionet.org/files/mimiciv/3.1/hosp/`. Only `admissions.csv.gz` is needed from MIMIC-IV. Section 0.2 can unpack the Google Drive zips that are currently inside the repository's `data/` folder.
3. **Model A on the RTX 5060 Ti** (16 GB) runs through a local [Ollama](https://ollama.com) server, so MIMIC text never leaves the machine (PhysioNet's LLM rule):
   `curl -fsSL https://ollama.com/install.sh | sh`, then `ollama pull qwen2.5:14b` (about 9 GB, fits in 16 GB VRAM with an 8k context). If Ollama runs on the Windows side instead of in WSL, set `OLLAMA_HOST` to the Windows address.
4. Run top to bottom. Without Model A the notebook takes about 2 minutes. Model A takes about 1 to 4 s per case on this GPU after a one-minute model load: the 25-case pilot takes about 2 minutes and the full 460-case validation run 15 to 30 minutes. The run resumes if it is interrupted.

> **Privacy.** Sections 3.2 to 3.4 show short, truncated samples of the record text, as the rubric asks. Do not commit this notebook with its outputs. Save the executed copy under `results/workbook2/` (gitignored), for example
> `jupyter nbconvert --to notebook --execute notebooks/workbook2.ipynb --output-dir results/workbook2 --ExecutePreprocessor.timeout=-1`
> and clear outputs before committing (`jupyter nbconvert --clear-output --inplace notebooks/workbook2.ipynb`). Set `SHOW_TEXT_SAMPLES = False` to hide all record text.

## 0. Setup

### 0.1 Configuration

Everything you might change is in this cell. Environment variables override the defaults.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

MIMIC_DIR = Path(os.environ.get("MIMIC_DIR", Path.home() / "data" / "mimic")).expanduser()
os.environ["MIMIC_DIR"] = str(MIMIC_DIR)              # must be set before the clinverify modules are imported

OUT = ROOT / "results" / "workbook2"                  # figures and aggregate tables for the docx (gitignored)
FIG = OUT / "figures"
WAREHOUSE_DB = MIMIC_DIR / "warehouse" / "clinverify.duckdb"   # holds patient data, so it stays next to the data
PREPARED_DIR = MIMIC_DIR / "prepared"                          # exported train/val/test datasets (patient data)
SPLIT_FILE = Path(os.environ.get("WB2_SPLIT_FILE", ROOT / "data" / "splits" / "ext_cdm_splits_v2.json"))

SHOW_TEXT_SAMPLES = True        # False hides all record text (samples show IDs and counts only)
UNPACK_DRIVE_ZIPS = False       # True: section 0.2 unpacks data/3.1-*.zip into MIMIC_DIR

# Model A (local LLM through Ollama)
RUN_MODEL_A = True
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:14b")
N_PILOT = 25                    # first run: this many validation cases (a quick check that the setup works)
RUN_FULL_VALIDATION = True
SEED = 298                      # protocol v1.1 generation seeds are 298, 299, 300
TEMPERATURE, TOP_P, MAX_NEW_TOKENS, NUM_CTX = 0.0, 1.0, 256, 8192

OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
print("repository :", ROOT)
print("MIMIC_DIR  :", MIMIC_DIR)
print("split file :", SPLIT_FILE)
print("outputs    :", OUT)

### 0.2 Optional: unpack the Google Drive download

The repository's `data/` folder holds a partial MIMIC-IV v3.1 download from Google Drive (`3.1-*.zip`, `labevents.csv-001.gz`, `chartevents.csv-002.gz`). `.gitignore` does not cover those names, so they should not stay in the repository. With `UNPACK_DRIVE_ZIPS = True` this cell unpacks them into `MIMIC_DIR/physionet.org/files/mimiciv/` and checks each file against PhysioNet's `SHA256SUMS.txt`. A file whose hash does not match is reported and not used. Delete the originals from `data/` afterwards.

In [ ]:
import hashlib
import shutil
import zipfile

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

if UNPACK_DRIVE_ZIPS:
    target = MIMIC_DIR / "physionet.org" / "files" / "mimiciv"          # the zips contain 3.1/...
    for z in sorted((ROOT / "data").glob("3.1-*.zip")):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)
        print("unpacked", z.name)
    base = target / "3.1"
    for src in (ROOT / "data").glob("*.csv-*.gz"):                      # Drive renamed labevents.csv.gz -> labevents.csv-001.gz
        name = src.name.split(".csv-")[0] + ".csv.gz"
        module = "icu" if name.startswith("chartevents") else "hosp"
        dest = base / module / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            shutil.copy2(src, dest)
        print("copied", src.name, "->", dest.relative_to(MIMIC_DIR))
    sums = dict(reversed(ln.split()) for ln in (base / "SHA256SUMS.txt").read_text().splitlines() if len(ln.split()) == 2)
    present = [p for p in sums if (base / p).exists()]
    bad = [p for p in present if sha256(base / p) != sums[p]]
    print(f"{len(present)} of {len(sums)} listed files present; {len(bad)} fail the checksum: {bad}")
    for p in bad:
        (base / p).rename((base / p).with_suffix(".incomplete"))
else:
    print("skipped (UNPACK_DRIVE_ZIPS = False)")

### 0.3 Imports and data check

In [ ]:
import json
import platform
import subprocess
import sys
import time
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from clinverify.data import ext_cdm as E
from clinverify.data import splits as S
from clinverify.data.case_loader import load_case_facts
from clinverify.eval.model_run import verify_model_plans
from clinverify.eval.protocol import load_protocol
from clinverify.models import model_a as A
from clinverify.rules.catalog import CATALOG_VERSION, catalog
from clinverify import warehouse as W

missing = [p for p in (E.EXT_CDM_DIR / "pathology_ids.json", E.MIMIC_IV_DIR / "hosp" / "admissions.csv.gz") if not p.exists()]
if missing:
    raise FileNotFoundError("Data not found. Expected:\n  " + "\n  ".join(map(str, missing)) +
                            "\nSet MIMIC_DIR, or unpack the download (section 0.2).")

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# Figure style: thin marks, recessive grid, text in ink colours. Pathology colours are fixed (never reassigned).
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
PATH_ORDER = ["appendicitis", "cholecystitis", "diverticulitis", "pancreatitis"]
PATH_COLOR = dict(zip(PATH_ORDER, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
SEQ = "#2a78d6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.edgecolor": INK_2, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False,
})

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    plt.show()

def save_table(df, name):
    # Aggregate tables only (never samples with record text)
    df.to_csv(OUT / f"{name}.csv", index=False)
    return df

def cut(text, n=80):
    text = str(text)
    return text if len(text) <= n else text[:n] + "…"

def sample(df, n=3, width=80):
    # First n rows; long text cut. With SHOW_TEXT_SAMPLES off, text columns are replaced by their length.
    out = df.head(n).copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].map(lambda v: cut(v, width)) if SHOW_TEXT_SAMPLES else out[c].astype(str).str.len().map(lambda n: f"<{n} chars>")
    return out

print("data found under", MIMIC_DIR)

## 3.1 Data process

The raw data are two credentialed PhysioNet datasets. Each team member who handles them has their own approved copy on their own machine. Only code, case-ID split lists and aggregate counts go to GitHub. The process has seven steps, each a function in `src/clinverify/data` that this notebook calls. The same functions feed the warehouse in 4.5.

In [ ]:
process = pd.DataFrame([
    ["1 Collect",      "PhysioNet download (per person, signed DUA)", "Ext-CDM v1.1 + MIMIC-IV v3.1 admissions",        "files on disk + SHA256 check",             "3.2"],
    ["2 Load",         "pandas read_csv",                             "9 Ext-CDM tables + lab mapping + pathology ids", "raw tables (raw layer)",                   "3.2, 4.5"],
    ["3 Pre-process",  "ext_cdm.clean_tables, validate_tables",       "raw tables",                                     "deduplicated, validated tables",           "3.3"],
    ["4 Transform",    "ext_cdm.transform_tables, case_facts, model_a.case_inputs", "cleaned tables",                   "parsed labs, case table, case facts, model input, TF-IDF matrix", "3.4"],
    ["5 Prepare",      "frozen split file (seed 298, grouped by patient)", "case table",                                "train 1,840 / validation 460 / test 100",  "3.5"],
    ["6 Store",        "DuckDB warehouse + data-quality gate",        "all of the above",                               "raw / staging / mart / ops schemas",       "4.5"],
    ["7 Use",          "Models A to E, Z3 verifier, evaluation harness", "mart tables",                                 "plans, verifier results, metrics",         "4.4"],
], columns=["step", "tool / code", "input", "output", "workbook section"])
save_table(process, "3_1_process")

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

def flow_diagram(boxes, arrows, size, name, title=None, groups=()):
    # boxes: {key: (x, y, w, h, title, subtitle, fill)}; arrows: [(from, to, label)]; groups: [(x, y, w, h, label)]
    fig, ax = plt.subplots(figsize=size)
    ax.axis("off"); ax.grid(False)
    for x, y, w, h, label in groups:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.15", fc="#f6f6f4", ec="#b9b8b2", lw=1, ls="--"))
        ax.text(x + 0.12, y + h - 0.12, label, ha="left", va="top", fontsize=9, color=INK_2, style="italic")
    for x, y, w, h, t, sub, fill in boxes.values():
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fill, ec=INK_2, lw=1))
        ax.text(x + w / 2, y + h * (0.66 if sub else 0.5), t, ha="center", va="center", fontsize=9.5, fontweight="bold")
        if sub:
            ax.text(x + w / 2, y + h * 0.3, sub, ha="center", va="center", fontsize=8, color=INK_2)
    def edge(box, toward):
        x, y, w, h = box[:4]; cx, cy = x + w / 2, y + h / 2
        tx, ty = toward
        if abs(tx - cx) * h > abs(ty - cy) * w:
            return (x + w if tx > cx else x, cy)
        return (cx, y + h if ty > cy else y)
    for a, b, label in arrows:
        A_, B_ = boxes[a], boxes[b]
        ca = (A_[0] + A_[2] / 2, A_[1] + A_[3] / 2); cb = (B_[0] + B_[2] / 2, B_[1] + B_[3] / 2)
        p, q = edge(A_, cb), edge(B_, ca)
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=11, color=INK_2, lw=1.1, shrinkA=2, shrinkB=2))
        if label:
            mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
            if abs(q[0] - p[0]) < abs(q[1] - p[1]):          # vertical arrow: label beside the line
                ax.text(mx + 0.1, my, label, ha="left", va="center", fontsize=7.5, color=INK_2)
            else:                                            # horizontal arrow: label above the line
                ax.text(mx, my + 0.08, label, ha="center", va="bottom", fontsize=7.5, color=INK_2)
    xs = [b[0] + b[2] for b in boxes.values()] + [g[0] + g[2] for g in groups]
    ys = [b[1] + b[3] for b in boxes.values()] + [g[1] + g[3] for g in groups]
    ax.set_xlim(min([b[0] for b in boxes.values()] + [g[0] for g in groups]) - 0.2, max(xs) + 0.2)
    ax.set_ylim(min([b[1] for b in boxes.values()] + [g[1] for g in groups]) - 0.2, max(ys) + 0.2)
    if title:
        ax.set_title(title, loc="left")
    save(fig, name)

DATA, PROC, STORE, MODEL = "#dae8fa", "#eef3e6", "#fdf0d5", "#f3e6ef"
flow_diagram({
    "src":   (0.0, 2.0, 2.2, 1.0, "1 Collect", "PhysioNet, SHA256", DATA),
    "load":  (2.7, 2.0, 2.2, 1.0, "2 Load", "raw tables", DATA),
    "pre":   (5.4, 2.0, 2.2, 1.0, "3 Pre-process", "dedupe, validate", PROC),
    "tr":    (8.1, 2.0, 2.2, 1.0, "4 Transform", "labs, facts, inputs", PROC),
    "prep":  (8.1, 0.2, 2.2, 1.0, "5 Prepare", "train / val / test", PROC),
    "wh":    (5.4, 0.2, 2.2, 1.0, "6 Store", "DuckDB warehouse", STORE),
    "use":   (2.7, 0.2, 2.2, 1.0, "7 Use", "models + verifier", MODEL),
}, [("src", "load", ""), ("load", "pre", ""), ("pre", "tr", ""), ("tr", "prep", ""), ("prep", "wh", ""), ("wh", "use", "")],
   (10, 3.4), "3_1_data_process", "Data process: from download to prepared datasets")

## 3.2 Data collection

### 3.2.1 Sources and parameters

In [ ]:
def dir_mb(p):
    return round(sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e6, 1) if Path(p).exists() else 0.0

ids = E.load_pathology_ids()
cases = E.case_table(ids)
hosp = E.MIMIC_IV_DIR / "hosp"
sources = pd.DataFrame([
    {"dataset": "MIMIC-IV-Ext-CDM", "version": "1.1", "publisher": "PhysioNet (Hager et al. 2024)",
     "licence": "PhysioNet Credentialed Health Data License 1.5.0", "role": "working set: 2,400 curated admissions",
     "files": len(list(E.EXT_CDM_DIR.glob("*.csv"))), "size_mb": dir_mb(E.EXT_CDM_DIR)},
    {"dataset": "MIMIC-IV", "version": "3.1", "publisher": "PhysioNet (Johnson et al. 2023)",
     "licence": "PhysioNet Credentialed Health Data License 1.5.0", "role": "hosp/admissions: hadm_id -> subject_id for patient-grouped splits",
     "files": len(list(hosp.glob("*.csv.gz"))), "size_mb": dir_mb(hosp)},
])
save_table(sources, "3_2_sources")

In [ ]:
parameters = pd.DataFrame([
    ["population", "adult hospital admissions with one of four abdominal pathologies, Beth Israel Deaconess Medical Center"],
    ["time span", "MIMIC-IV admissions 2008-2022 (dates shifted for de-identification)"],
    ["unit of analysis", "one hospital admission (hadm_id); a patient (subject_id) can have several"],
    ["label", "pathology from pathology_ids.json: " + ", ".join(f"{p} {len(h):,}" for p, h in ids.items())],
    ["inputs a model may see", "history of present illness, physical exam, laboratory tests, microbiology, radiology reports"],
    ["withheld from models", "discharge diagnosis, ICD diagnoses and procedures, discharge procedures (they give the answer away; used only for case facts)"],
    ["access", "CITI training + PhysioNet credentialing + signed DUA, per person; no hosted LLM APIs"],
], columns=["parameter", "value"])
parameters

### 3.2.2 Integrity and quantity

PhysioNet ships a `SHA256SUMS.txt` with every dataset. Every Ext-CDM file is hashed and compared. The quantity table is computed from the files on disk.

In [ ]:
listed = [ln.split() for ln in (E.EXT_CDM_DIR / "SHA256SUMS.txt").read_text().replace("\r", "").splitlines() if len(ln.split()) == 2]
bad = [rel for digest, rel in listed if not (E.EXT_CDM_DIR / rel).exists() or sha256(E.EXT_CDM_DIR / rel) != digest]
print(f"Ext-CDM checksum: {len(listed) - len(bad)} of {len(listed)} files match" + (f"; mismatched: {bad}" if bad else ""))

raw = E.load_tables()
raw["lab_test_mapping"] = pd.read_csv(E.EXT_CDM_DIR / "lab_test_mapping.csv")
quantity = pd.DataFrame([{
    "table": name, "file_mb": round((E.EXT_CDM_DIR / f"{name}.csv").stat().st_size / 1e6, 2),
    "rows": len(df), "columns": df.shape[1],
    "cases": int(df["hadm_id"].nunique()) if "hadm_id" in df else None,
    "column names": ", ".join(df.columns),
} for name, df in raw.items()])
adm = pd.read_csv(hosp / "admissions.csv.gz", usecols=["subject_id", "hadm_id"])
linked = int(cases["hadm_id"].isin(adm["hadm_id"]).sum())
print(f"Raw rows: {quantity['rows'].sum():,} in {len(quantity)} tables | cases: {len(cases):,} | "
      f"linked to MIMIC-IV admissions: {linked:,} of {len(cases):,} | patients: {adm[adm.hadm_id.isin(cases.hadm_id)].subject_id.nunique():,}")
save_table(quantity, "3_2_quantity")

### 3.2.3 Samples from the raw data

The first three rows of each raw table as read from disk. Text is cut to 80 characters. `___` in the notes is MIMIC's de-identification, not missing data.

In [ ]:
for name, df in raw.items():
    display(Markdown(f"**{name}** — {len(df):,} rows × {df.shape[1]} columns"))
    display(sample(df))

## 3.3 Data pre-processing

Tools: pandas for cleaning and `ext_cdm.validate_tables` for validation. The validation runs 7 kinds of check: unique labels, required columns, one row per case in case-level tables, no unknown or empty `hadm_id`, full coverage of the tables every case must have, and no empty history text. The same checks gate the warehouse load in 4.5.

### 3.3.1 Cleaning: exact duplicates

In [ ]:
cleaned, dup_report = E.clean_tables(raw | {})
cleaned.pop("lab_test_mapping", None)
dup_report = dup_report[dup_report["table"] != "lab_test_mapping"]
print(f"Duplicate rows removed: {int(dup_report['duplicate_rows'].sum())}")
save_table(dup_report, "3_3_duplicates")

### 3.3.2 Missing values and how each is handled

Columns with no missing values are left out. Reference ranges stay missing and are flagged `unknown` later. They are never imputed, because a reference range is a fact about one lab, not a statistic.

In [ ]:
missing_tbl = E.missing_report({**cleaned, "lab_test_mapping": raw["lab_test_mapping"]})
save_table(missing_tbl, "3_3_missing")

### 3.3.3 Validation: passes on the cleaned tables, fails on a deliberately corrupted copy

The corrupted copy duplicates three cases, empties ten history notes, drops the `hadm_id` column from labs, plants an unknown case id in microbiology and removes radiology for 1% of cases. Every problem must be caught.

In [ ]:
problems_good = E.validate_tables(cleaned, ids)
problems_bad = E.validate_tables(E.corrupt(cleaned), ids)
print("cleaned tables :", "PASS" if not problems_good else f"FAIL {problems_good}")
print("corrupted copy :", f"FAIL as expected, {len(problems_bad)} problems caught" if problems_bad else "PASS (unexpected)")
save_table(pd.DataFrame({"problem caught on the corrupted copy": problems_bad}), "3_3_validation_failures")

### 3.3.4 Samples from the pre-processed data

In [ ]:
for name in ("history_of_present_illness", "laboratory_tests", "radiology_reports"):
    display(Markdown(f"**{name}** (cleaned) — {len(cleaned[name]):,} rows"))
    display(sample(cleaned[name]))

## 3.4 Data transformation

Tools: pandas (`ext_cdm.transform_tables`), regular expressions over ICD titles (`case_facts.py`), the model-input builder (`model_a.case_inputs`) and scikit-learn (`TfidfVectorizer`). Four target formats come out of this step:

1. **Normalised tables**: text trimmed and single-spaced, with a word count. Lab strings are split into number, unit and `<`/`>` qualifier and flagged low, normal, high or unknown against the row's own reference range.
2. **Case table** (`cases_master`): one row per admission with its label and counts per table.
3. **Model input** (one text per case): the record sections a model may see, in a fixed order with fixed length limits. Models A and E read exactly this text.
4. **Feature matrix**: TF-IDF of the model input, used by Model E.

### 3.4.1 Transformation log

In [ ]:
transformed = E.transform_tables(cleaned)
save_table(pd.DataFrame(E.TRANSFORM_LOG, columns=["table", "input", "operation", "output", "why"]), "3_4_transform_log")

### 3.4.2 Lab strings before and after parsing

In [ ]:
labs = transformed["laboratory_tests"]
show = labs[["hadm_id", "label", "valuestr", "value", "unit", "qualifier", "ref_range_lower", "ref_range_upper", "flag"]]
examples = pd.concat([show[show["flag"] == f].head(2) for f in ("low", "normal", "high", "unknown")] +
                     [show[show["qualifier"].notna()].head(2)])
display(examples)
flag_counts = labs["flag"].value_counts().rename_axis("flag").reset_index(name="rows")
flag_counts["share_%"] = (flag_counts["rows"] / len(labs) * 100).round(1)
save_table(flag_counts, "3_4_lab_flags")

### 3.4.3 Case table and case facts

`cases_master` is the case-level table. The case facts come from ICD titles and the discharge diagnosis text. The rules use them, and they are **never** shown to a model. In strict mode, a fact the record does not establish stays unknown.

In [ ]:
master = transformed["cases_master"]
display(master.head(5))
import case_facts_counts as CF
facts = load_case_facts()                       # strict mode: an unestablished fact stays unknown
rows = []
for path, names in CF.FACTS.items():
    derived = [c["facts"] for c in facts.values() if c["pathology"] == path]
    for f in names:
        t, fl = sum(d.get(f) is True for d in derived), sum(d.get(f) is False for d in derived)
        rows.append({"pathology": path, "cases": len(derived), "fact": f, "true": t, "false": fl, "unknown": len(derived) - t - fl})
save_table(pd.DataFrame(rows), "3_4_case_facts")

### 3.4.4 Model input format

One case's model input, as the models receive it. The section order and limits are fixed: labs (abnormal first, at most 40), microbiology (at most 15), and up to 3 radiology reports of at most 1,500 characters each.

In [ ]:
inputs = A.case_inputs(transformed)
lengths = pd.Series({h: len(t) for h, t in inputs.items()}, name="input_chars")
first = next(iter(inputs))
print(f"{len(inputs):,} model inputs | median {lengths.median():,.0f} characters, 95th percentile {lengths.quantile(.95):,.0f}")
print("\n--- case", first, "(first 900 characters) ---\n")
print(inputs[first][:900] + " …" if SHOW_TEXT_SAMPLES else "<hidden: SHOW_TEXT_SAMPLES = False>")

### 3.4.5 Feature matrix for the classical model

The vocabulary is fit on the **training split only** (section 3.5), so no validation or test text leaks into the features.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

split_data = json.loads(SPLIT_FILE.read_text())
SPLITS = {k: [int(h) for h in v] for k, v in split_data["splits"].items()}
label_of = cases.set_index("hadm_id")["pathology"]

vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z]+\b", ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True, max_features=50000)
X = {k: None for k in SPLITS}
X["train"] = vectorizer.fit_transform([inputs[h] for h in SPLITS["train"]])
for k in ("val", "test"):
    X[k] = vectorizer.transform([inputs[h] for h in SPLITS[k]])
y = {k: np.array([label_of[h] for h in v]) for k, v in SPLITS.items()}
pd.DataFrame([{"split": k, "rows (cases)": m.shape[0], "features": m.shape[1], "non-zero share_%": round(m.nnz / (m.shape[0] * m.shape[1]) * 100, 3)}
              for k, m in X.items()])

## 3.5 Data preparation

The split is fixed and versioned (`data/splits/ext_cdm_splits_v2.json`, split v2). It holds 100 test cases (the held-out set named in the abstract), and 20% of the remaining cases go to validation. It is stratified by pathology with seed 298 and **grouped by patient**, so all of one patient's admissions land in the same split. Version 1 split by admission and let 5 patients cross splits. Version 2 has none. The file's SHA-256 is pinned in the evaluation protocol, so a changed split is refused at scoring time.

In [ ]:
protocol = load_protocol(ROOT)
frozen = SPLIT_FILE.resolve() == (ROOT / protocol["split_file"]).resolve()
if frozen:
    ok = sha256(SPLIT_FILE) == protocol["split_sha256"]
    print("split file SHA-256 matches the frozen protocol:", ok)
    assert ok, "the split file changed (line endings count): restore it with  git checkout -- data/splits/ext_cdm_splits_v2.json"
subject_of = E.subject_map()
leaks = S.leakage_check(SPLITS, cases) + S.patient_leakage_check(SPLITS, subject_of)
print("leakage checks:", "PASS (no case and no patient in two splits)" if not leaks else leaks)

split_rows = []
for name, hs in SPLITS.items():
    counts = label_of[hs].value_counts()
    split_rows.append({"split": name, "cases": len(hs), "patients": subject_of.reindex(hs).nunique(),
                       **{p: int(counts.get(p, 0)) for p in PATH_ORDER},
                       "share_%": round(len(hs) / len(cases) * 100, 1)})
split_table = pd.DataFrame(split_rows)
save_table(split_table, "3_5_split_table")

### 3.5.1 Prepared datasets

Each split is exported as one table, with one row per case: id, label, the model-input text and its size. Models A and E read these. The files hold patient text, so they are written next to the data (`MIMIC_DIR/prepared/`), not into the repository.

In [ ]:
import duckdb

prepared = master[["hadm_id", "pathology", "hpi_words", "n_laboratory_tests", "n_radiology_reports", "n_microbiology"]].copy()
prepared["subject_id"] = prepared["hadm_id"].map(subject_of)
prepared["input_chars"] = prepared["hadm_id"].map(lengths)
prepared["split"] = prepared["hadm_id"].map({h: k for k, hs in SPLITS.items() for h in hs})
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
for name in SPLITS:
    part = prepared[prepared["split"] == name].assign(model_input=lambda d: d["hadm_id"].map(inputs))
    con.register("part", part)
    con.execute(f"COPY part TO '{PREPARED_DIR / f'{name}.parquet'}' (FORMAT PARQUET)")
    con.unregister("part")
con.close()
print("written:", ", ".join(f"{p.name} ({p.stat().st_size / 1e6:.1f} MB)" for p in sorted(PREPARED_DIR.glob("*.parquet"))))
for name in SPLITS:
    display(Markdown(f"**{name}** — {int((prepared['split'] == name).sum()):,} cases (first 3; model text not shown)"))
    display(prepared[prepared["split"] == name].drop(columns="subject_id").head(3))

## 3.6 Data statistics

### 3.6.1 Summary by stage

In [ ]:
case_tables = E.CASE_LEVEL + E.EVENT_LEVEL
stage_summary = pd.DataFrame([
    ["Raw", len(case_tables) + 1, sum(len(raw[t]) for t in case_tables) + len(raw["lab_test_mapping"]),
     sum(raw[t].shape[1] for t in case_tables), len(cases), "files as downloaded"],
    ["Pre-processed", len(case_tables), sum(len(cleaned[t]) for t in case_tables), sum(cleaned[t].shape[1] for t in case_tables),
     len(cases), f"{int(dup_report['duplicate_rows'].sum())} exact duplicates removed; validation passes"],
    ["Transformed", len(case_tables) + 1, sum(len(transformed[t]) for t in case_tables) + len(master),
     sum(transformed[t].shape[1] for t in case_tables) + master.shape[1], len(master),
     "labs parsed and flagged, text normalised, case table, model inputs, TF-IDF features"],
    ["Prepared", len(SPLITS), len(prepared), prepared.shape[1] + 1, len(prepared),
     " / ".join(f"{k} {len(v):,}" for k, v in SPLITS.items()) + ", no patient in two splits"],
], columns=["stage", "tables", "rows", "columns", "cases", "what changed"])
save_table(stage_summary, "3_6_stage_summary")

### 3.6.2 Figures

In [ ]:
# (a) class balance in each split: grouped bars, one colour per pathology (fixed order), direct labels
fig, ax = plt.subplots(figsize=(8, 3.6))
width = 0.2
xs = np.arange(len(SPLITS))
for i, p in enumerate(PATH_ORDER):
    vals = [int(label_of[hs].eq(p).sum()) for hs in SPLITS.values()]
    shares = [v / len(hs) * 100 for v, hs in zip(vals, SPLITS.values())]
    bars = ax.bar(xs + (i - 1.5) * width, shares, width * 0.92, color=PATH_COLOR[p], label=p)
    ax.bar_label(bars, labels=[f"{v}" for v in vals], fontsize=7.5, color=INK_2, padding=2)
ax.set_xticks(xs, [f"{k}\n({len(v):,} cases)" for k, v in SPLITS.items()])
ax.set_ylabel("share of split (%)")
ax.set_title("Each split keeps the same pathology mix (labels: cases)", loc="left")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.grid(axis="x", visible=False)
save(fig, "3_6a_class_balance_by_split")

In [ ]:
# (b) missing values per column (only columns with any)
m = missing_tbl.assign(col=lambda d: d["table"] + "." + d["column"]).sort_values("missing_pct")
fig, ax = plt.subplots(figsize=(8, 0.35 * len(m) + 1))
bars = ax.barh(m["col"], m["missing_pct"], color=SEQ, height=0.6)
ax.bar_label(bars, labels=[f"{p:.1f}% ({n:,})" for p, n in zip(m["missing_pct"], m["missing"])], fontsize=8, color=INK_2, padding=3)
ax.set_xlabel("missing (% of rows)")
ax.set_xlim(0, max(100, m["missing_pct"].max() * 1.25) if len(m) else 100)
ax.set_title("Missing values by column (kept as missing, not imputed)", loc="left")
ax.grid(axis="y", visible=False)
save(fig, "3_6b_missing_values")

In [ ]:
# (c) lab flags by pathology: 100% stacked bars; low/high are the two poles, normal neutral, unknown light with hatch
flag_order = ["low", "normal", "high", "unknown"]
flag_color = {"low": "#2a78d6", "normal": "#c9c8c2", "high": "#eb6834", "unknown": "#f1f0ec"}
share = labs.merge(master[["hadm_id", "pathology"]], on="hadm_id").groupby(["pathology", "flag"]).size().unstack(fill_value=0)
share = (share.div(share.sum(axis=1), axis=0) * 100).reindex(PATH_ORDER)[flag_order]
fig, ax = plt.subplots(figsize=(8, 3.2))
left = np.zeros(len(share))
for f in flag_order:
    bars = ax.barh(share.index, share[f], left=left, color=flag_color[f], edgecolor="white", linewidth=2, height=0.6,
                   hatch="///" if f == "unknown" else None, label=f)
    ax.bar_label(bars, labels=[f"{v:.0f}%" if v >= 6 else "" for v in share[f]], label_type="center", fontsize=8, color=INK)
    left += share[f].values
ax.set_xlim(0, 100); ax.set_xlabel("share of lab rows (%)")
ax.invert_yaxis()
ax.set_title("Lab results by flag and pathology", loc="left", pad=24)
ax.legend(ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8.5)
ax.grid(False)
save(fig, "3_6c_lab_flags_by_pathology")

In [ ]:
# (d) model-input length by pathology
fig, ax = plt.subplots(figsize=(8, 3.4))
data = [prepared.loc[prepared["pathology"] == p, "input_chars"] for p in PATH_ORDER]
bp = ax.boxplot(data, orientation="horizontal", widths=0.5, patch_artist=True, showfliers=False, medianprops=dict(color=INK, lw=1.5))
for patch, p in zip(bp["boxes"], PATH_ORDER):
    patch.set(facecolor=PATH_COLOR[p], alpha=0.85, edgecolor=INK_2)
for i, d in enumerate(data, 1):
    ax.text(d.quantile(0.75) + 60, i + 0.3, f"median {d.median():,.0f}", fontsize=8, color=INK_2)
ax.set_yticks(range(1, 5), PATH_ORDER)
ax.invert_yaxis()
ax.set_xlabel("characters in the model input")
ax.set_title("Model input length by pathology (outliers not drawn)", loc="left")
ax.grid(axis="y", visible=False)
save(fig, "3_6d_input_length")

In [ ]:
# (e) what the record facts let each rule check, on all covered cases (strict mode)
cov = CF.rule_eligibility(facts)
cov_cols = [c for c in ("testable_on_facts", "unknown_fact", "condition_false") if c in cov.columns]
cov_plot = cov.set_index("rule")[cov_cols]
cov_pct = cov_plot.div(cov_plot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
colors = {"testable_on_facts": "#2a78d6", "unknown_fact": "#f1f0ec", "condition_false": "#c9c8c2"}
names = {"testable_on_facts": "testable", "unknown_fact": "needed fact unknown", "condition_false": "rule does not apply"}
fig, ax = plt.subplots(figsize=(8, 0.38 * len(cov_pct) + 1.3))
left = np.zeros(len(cov_pct))
for c in cov_cols:
    bars = ax.barh(cov_pct.index, cov_pct[c], left=left, color=colors[c], edgecolor="white", linewidth=2, height=0.65,
                   hatch="///" if c == "unknown_fact" else None, label=names[c])
    ax.bar_label(bars, labels=[f"{n:,}" if p >= 8 else "" for n, p in zip(cov_plot[c], cov_pct[c])], label_type="center", fontsize=7.5)
    left += cov_pct[c].values
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("share of the rule's cases (%), labels: cases")
ax.set_title("Which rules the record facts can test", loc="left", pad=24)
ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8.5)
ax.grid(False)
save(fig, "3_6e_rule_coverage")
save_table(cov.reset_index(drop=True), "3_6_rule_coverage")

In [ ]:
# (f) rows removed by pre-processing, per table
removed = dup_report.set_index("table")["duplicate_rows"]
dropped_proc = len(cleaned["discharge_procedures"]) - len(transformed["discharge_procedures"])
removed_tbl = pd.DataFrame({"duplicates": removed, "empty rows dropped": 0}).fillna(0)
removed_tbl.loc["discharge_procedures", "empty rows dropped"] = dropped_proc
removed_tbl = removed_tbl[(removed_tbl.sum(axis=1) > 0)].sort_values("duplicates")
fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(removed_tbl), 2) + 1))
b1 = ax.barh(removed_tbl.index, removed_tbl["duplicates"], color=SEQ, height=0.6, label="exact duplicates")
b2 = ax.barh(removed_tbl.index, removed_tbl["empty rows dropped"], left=removed_tbl["duplicates"], color="#c9c8c2",
             edgecolor="white", linewidth=2, height=0.6, label="empty rows dropped")
ax.bar_label(b2, labels=[f"{int(a + b):,}" for a, b in zip(removed_tbl["duplicates"], removed_tbl["empty rows dropped"])], padding=3, fontsize=8, color=INK_2)
ax.set_xlabel("rows removed")
ax.set_title("Rows removed by pre-processing", loc="left")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
save(fig, "3_6f_rows_removed")

## 4.1 Model proposals

Five models are proposed, plus the reproduced published baseline. Models A and E are built and run in 298A (section 4.4); B, C and D come in 298B. **Model C is the project's new contribution**: a solver-in-the-loop repair cycle that uses team-written clinical rules. No paper in our reviewed set does this for clinical plans. **Model E** is a proposal added in this workbook, a cheap statistical reference point that shows how much an LLM adds. The team should confirm it.

| # | Model | Concept | Architecture / algorithm | Features (input) | Output | Phase |
|---|---|---|---|---|---|---|
| A | **LLM plan generator** | an LLM writes the diagnosis and the plan directly; the verifier checks it afterwards | decoder-only transformer, Qwen2.5-14B-Instruct (4-bit, local), greedy decoding, output constrained to the plan JSON schema | model-input text (3.4.4) | structured plan | 298A (built) |
| B | **LLM to formal specification** | the LLM translates the case into a formal specification; a solver does the reasoning | LLM translation into SMT-LIB / Z3 facts + Z3 solve (Logic-LM, SatLM pattern) | model-input text | facts + plan derived by the solver | 298B |
| C | **Solver-in-the-loop repair** (innovative) | Model A, but each rejected plan goes back to the LLM with the broken rule's ID and message, for at most k rounds | LLM + Z3 verifier as an external critic; feedback from rule violation messages (LLM-Modulo) | model-input text + verifier feedback | repaired plan + repair trace | 298B |
| D | **LLM + classical planner** | the LLM maps the case to a PDDL problem; a classical planner builds a plan that satisfies the domain | LLM translation + Fast Downward on a hand-written clinical PDDL domain (LLM+P) | model-input text + domain model | plan valid by construction | 298B |
| E | **Classical ML + guideline template** | a text classifier predicts the diagnosis; a fixed per-diagnosis template gives the treatment | TF-IDF (1-2-grams) + multinomial logistic regression; treatment template per diagnosis | TF-IDF matrix (3.4.5) | diagnosis with class probabilities + template plan | 298A (built) |
| — | Reproduced baseline | Hager et al. (2024), autonomous clinical decision making | OASST 70B (Llama 2 base) in the authors' framework | interactive information requests | diagnosis | 298A (needs a larger GPU) |

The majority-class predictor (always "appendicitis") is reported next to them as the floor every model has to beat.

## 4.2 Model supports

### 4.2.1 Platform, environment and tools (measured on this machine)

In [ ]:
def run_cmd(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=20).stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        return ""

gpu = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
client = A.Ollama(OLLAMA_MODEL)
try:
    ollama_version = json.loads(subprocess.run(["curl", "-s", client.host + "/api/version"], capture_output=True, text=True, timeout=5).stdout)["version"]
except Exception:
    ollama_version = "not reachable at " + client.host
pkgs = ["pandas", "numpy", "scikit-learn", "duckdb", "z3-solver", "matplotlib"]
env = pd.DataFrame(
    [["OS", platform.platform()], ["CPU", platform.processor() or platform.machine()], ["Python", platform.python_version()],
     ["GPU", gpu or "none detected"], ["Ollama server", ollama_version], ["Model A weights", OLLAMA_MODEL],
     ["git commit", run_cmd(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"])]]
    + [[p, version(p)] for p in pkgs], columns=["component", "version / value"])
save_table(env, "4_2_environment")

In [ ]:
supports = pd.DataFrame([
    ["A", "local GPU (RTX 5060 Ti, 16 GB)", "Ollama (llama.cpp, CUDA), Qwen2.5-14B-Instruct Q4_K_M", "JSON-schema-constrained decoding, Z3 verifier, evaluation harness"],
    ["B", "local GPU", "Ollama + z3-solver", "SMT-LIB parser, translation-fidelity check"],
    ["C", "local GPU", "Ollama + z3-solver", "repair loop, round limit, cost logging (tokens, solver ms)"],
    ["D", "local GPU + CPU", "Ollama + Fast Downward planner", "hand-written PDDL domain, plan validator (VAL)"],
    ["E", "CPU", "scikit-learn (TfidfVectorizer, LogisticRegression)", "template plans, Z3 verifier"],
    ["all", "local machine; GitHub for code only", "Python, pandas, DuckDB warehouse, pytest, GitHub Actions CI", "Linear for tasks; Jupyter for demos"],
], columns=["model", "hardware", "framework / runtime", "other tools"])
supports

### 4.2.2 Platform architecture

In [ ]:
flow_diagram({
    "data":  (0.3, 3.6, 2.4, 1.0, "PhysioNet data", "Ext-CDM 1.1, MIMIC-IV 3.1", DATA),
    "wh":    (3.7, 3.6, 2.4, 1.0, "DuckDB warehouse", "raw / staging / mart / ops", STORE),
    "prep":  (7.1, 3.6, 2.4, 1.0, "Prepared splits", "train / val / test", PROC),
    "ml":    (10.5, 3.6, 2.4, 1.0, "scikit-learn", "Model E (CPU)", MODEL),
    "llm":   (7.1, 1.9, 2.4, 1.0, "Ollama on GPU", "Models A-D (local LLM)", MODEL),
    "z3":    (10.5, 1.9, 2.4, 1.0, "Z3 verifier", "rule catalog v2.1 (12 rules)", PROC),
    "eval":  (13.9, 1.9, 2.4, 1.0, "Evaluation harness", "metrics, protocol v1.1", PROC),
    "facts": (10.5, 0.2, 2.4, 1.0, "Case facts", "ICD + discharge text", PROC),
    "gh":    (13.9, -1.6, 2.4, 1.0, "GitHub + CI", "code, split IDs, counts", "#ffffff"),
}, [("data", "wh", "ingest"), ("wh", "prep", ""), ("prep", "llm", "case text"), ("prep", "ml", "features"),
    ("llm", "z3", "plan JSON"), ("ml", "z3", "plan"), ("facts", "z3", "never shown to models"),
    ("z3", "eval", "results"), ("eval", "gh", "aggregate counts only")],
   (15, 6.4), "4_2_platform_architecture", "Platform architecture: everything with patient data stays on the local machine",
   groups=[(0.0, -0.1, 16.6, 5.05, "local machine (credentialed data)")])

### 4.2.3 ML data flow for the two models built in 298A

In [ ]:
flow_diagram({
    "in":   (0.0, 1.0, 2.2, 1.0, "Model input", "text per case (3.4.4)", DATA),
    "p":    (2.7, 2.0, 2.2, 1.0, "Prompt", "system + schema", MODEL),
    "g":    (5.4, 2.0, 2.2, 1.0, "LLM decode", "greedy, seed 298", MODEL),
    "j":    (8.1, 2.0, 2.2, 1.0, "Parse JSON", "plan or missing", PROC),
    "t":    (2.7, 0.0, 2.2, 1.0, "TF-IDF", "fit on train only", MODEL),
    "lr":   (5.4, 0.0, 2.2, 1.0, "Logistic reg.", "class probabilities", MODEL),
    "tp":   (8.1, 0.0, 2.2, 1.0, "Template", "plan per diagnosis", PROC),
    "v":    (10.8, 1.0, 2.2, 1.0, "Z3 verifier", "+ record facts", PROC),
    "m":    (13.5, 1.0, 2.0, 1.0, "Metrics", "4.4", PROC),
}, [("in", "p", "Model A"), ("p", "g", ""), ("g", "j", ""), ("j", "v", ""), ("in", "t", "Model E"), ("t", "lr", ""),
    ("lr", "tp", ""), ("tp", "v", ""), ("v", "m", "")], (12, 3.2), "4_2_ml_data_flow", "ML data flow: Model A (top) and Model E (bottom)")

## 4.3 Model comparison and justification

| | A: LLM generator | B: LLM → formal spec | C: solver-in-the-loop | D: LLM + planner | E: classical ML |
|---|---|---|---|---|---|
| **Targeted problem** | can an LLM write a plan that passes the rules without help? | does moving the reasoning into a solver make plans faithful to the case? | can explicit rule feedback turn failing plans into passing ones? | can plans be valid by construction? | how much does an LLM add over a cheap classifier? |
| **Features** | full record text | full record text | record text + violated rule ID and message | record text + PDDL domain | TF-IDF 1-2-grams (up to 50k) |
| **Approach** | deep learning (transformer LLM, prompting) | deep learning + symbolic (SMT) | deep learning + symbolic critic | deep learning + classical planning | statistics / classical ML |
| **Strengths** | simplest; no translation step; closest to the baseline paper | the explanation is the solver's proof; inconsistencies are caught | uses the rule catalog directly and names the broken rule; verifier feedback beat LLM self-critique 88/100 vs 55/100 (Valmeekam et al. 2023a) | planner output always satisfies the domain; plans are auditable | fast (seconds on a CPU), deterministic, gives calibrated probabilities and ROC curves, easy to explain |
| **Limitations** | no correction step; LLMs skip guideline treatment (Hager et al. 2024); output can be unparseable | translation errors (measured as translation fidelity); the formal language must cover each disease | can only repair what the rules cover; unknown facts give no feedback; extra generations cost time | needs a clinical PDDL domain, which does not exist and is hard to write; conditional lab rules fit actions badly | bag of words ignores negation and order; treatment is a fixed template, not reasoning about the case |
| **Justification** | the reference every other model is compared against; R10 requires it | tests the Logic-LM / SatLM claim in a clinical setting | the project's main hypothesis (Kambhampati et al. 2024, LLM-Modulo) | tests the LLM+P claim; shows where planning formalisms break | a statistical floor: if E matches A on diagnosis, the LLM's value must come from the plan, not the label |

All five are scored on the same fixed splits with the same verifier and the same metrics, so only the method changes between them. The Hager baseline is kept outside the five as the published reference. It measures autonomous diagnosis, which is a different task from plan validity.

## 4.4 Model evaluation methods

Definitions follow `docs/metrics.md` (protocol v1.1, frozen October 23, 2026). Every model is scored on the **validation split** here. The 100-case test split stays untouched until the team freezes Model A's settings and runs the formal test (`scripts/verify_model_plans.py --cohort test --generation-metadata …`).

| Metric | Type | Models | Meaning |
|---|---|---|---|
| diagnosis accuracy, per-pathology accuracy | classification | all | share of cases with the right pathology |
| macro precision / recall / F1, confusion matrix | classification | all | balanced over the four classes (appendicitis is 40% of cases) |
| ROC-AUC (one-vs-rest), log loss | probabilistic | E (models with class probabilities) | ranking quality and calibration; an LLM's greedy output has no probabilities |
| supported checked acceptance rate | verifier (primary, descriptive) | A–E | correct diagnosis + plan accepted + at least one decided blocking clinical rule, over covered cases |
| plan status mix, violations by rule | verifier | A–E | valid / violated / insufficient information, and which rules fail |
| unparseable / missing output rate | robustness | A–D | outputs that cannot be scored (counted as failures) |
| latency per case, tokens per second, solver ms | cost | A–E | what a run costs |
| escape rate ≤ 0.05, adjudicated validity (goal 0.85) | human-labelled | A–E | need independent plan labels; reported as unavailable until they exist |
| repair success rate, translation fidelity | secondary | C, B | 298B |

### 4.4.1 Scoring helpers

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score, log_loss,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)

VAL_IDS = SPLITS["val"]
val_facts = load_case_facts(VAL_IDS)
RESULTS = {}

def diagnosis_metrics(name, ids_, pred):
    truth = [label_of[h] for h in ids_]
    pred = [pred.get(h, "missing") for h in ids_]
    p, r, f, _ = precision_recall_fscore_support(truth, pred, labels=PATH_ORDER, average="macro", zero_division=0)
    per = {f"acc_{c}": np.mean([pp == c for t, pp in zip(truth, pred) if t == c]) for c in PATH_ORDER}
    return {"model": name, "cases": len(ids_), "accuracy": accuracy_score(truth, pred), "macro_precision": p,
            "macro_recall": r, "macro_f1": f, **per}

def verifier_metrics(plans, ids_, fixed_cohort=True):
    summary = verify_model_plans(plans, val_facts, catalog, expected_case_ids=ids_ if fixed_cohort else None,
                                 supported_diagnoses=protocol["treatment_diagnoses"], catalog_version=CATALOG_VERSION)
    keep = ["supported_checked_acceptance_rate", "n_supported_cases", "n_supported_checked_accepted",
            "joint_success_rate", "validity_rate", "n_missing_outputs", "n_unparseable", "solver_ms_mean"]
    return {k: summary[k] for k in keep}, summary

def confusion_plot(ids_, pred, title, name):
    truth = [label_of[h] for h in ids_]
    labels = PATH_ORDER + (["missing"] if any(h not in pred for h in ids_) else [])
    cm = confusion_matrix(truth, [pred.get(h, "missing") for h in ids_], labels=labels)[: len(PATH_ORDER)]
    fig, ax = plt.subplots(figsize=(4.8 + 0.6 * (len(labels) - 4), 4))
    ax.imshow(cm, cmap="Blues", vmin=0)
    ax.grid(False)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9, color="white" if cm[i, j] > cm.max() * 0.6 else INK)
    ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right"); ax.set_yticks(range(len(PATH_ORDER)), PATH_ORDER)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(title, loc="left")
    save(fig, name)

### 4.4.2 Majority class (floor)

In [ ]:
majority = pd.Series(y["train"]).value_counts().idxmax()
TEMPLATE = {   # Model E's fixed treatment per diagnosis. A software baseline, NOT a clinical recommendation.
    "appendicitis":   {"treatments": ["appendectomy", "preoperative_antibiotic"], "postop_antibiotic_days": 0},
    "cholecystitis":  {"treatments": ["cholecystectomy_during_admission"], "postop_antibiotic_days": None},
    "pancreatitis":   {"treatments": [], "postop_antibiotic_days": None},
    "diverticulitis": {"treatments": ["colonoscopy_after_recovery"], "postop_antibiotic_days": None},
}
def template_plans(pred):
    return [{"case_id": h, "diagnosis": d, **TEMPLATE[d]} for h, d in pred.items()]

pred_major = {h: majority for h in VAL_IDS}
ver_major, _ = verifier_metrics(template_plans(pred_major), VAL_IDS)
RESULTS["Majority class"] = {**diagnosis_metrics("Majority class", VAL_IDS, pred_major), **ver_major}
print("always predicts:", majority)
pd.Series(RESULTS["Majority class"]).to_frame("value")

### 4.4.3 Model E: TF-IDF + logistic regression

The regularisation strength C is chosen by 5-fold cross-validation on the **training split** (macro F1). The validation split is used once, for the reported numbers.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

t0 = time.perf_counter()
search = GridSearchCV(LogisticRegression(max_iter=3000, class_weight="balanced"),
                      {"C": [0.3, 1, 3, 10, 30]}, scoring="f1_macro", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
search.fit(X["train"], y["train"])
clf = search.best_estimator_
train_s = time.perf_counter() - t0
t0 = time.perf_counter()
proba = clf.predict_proba(X["val"])
pred_lr = dict(zip(VAL_IDS, clf.classes_[proba.argmax(1)]))
infer_ms = (time.perf_counter() - t0) / len(VAL_IDS) * 1000

cv = pd.DataFrame(search.cv_results_)[["param_C", "mean_test_score", "std_test_score"]].rename(
    columns={"param_C": "C", "mean_test_score": "cv macro F1 (train)", "std_test_score": "std"})
display(cv.round(3))
y_val_bin = np.array([[t == c for c in clf.classes_] for t in y["val"]])
ver_e, _ = verifier_metrics(template_plans(pred_lr), VAL_IDS)
RESULTS["Model E (TF-IDF + LR)"] = {**diagnosis_metrics("Model E (TF-IDF + LR)", VAL_IDS, pred_lr), **ver_e,
                                    "roc_auc_macro_ovr": roc_auc_score(y_val_bin, proba, average="macro"),
                                    "log_loss": log_loss(y["val"], proba, labels=clf.classes_),
                                    "train_seconds": train_s, "ms_per_case": infer_ms}
print(f"best C = {search.best_params_['C']}, train {train_s:.1f} s, {infer_ms:.3f} ms per case")
print(classification_report(y["val"], clf.classes_[proba.argmax(1)], labels=PATH_ORDER, digits=3, zero_division=0))

In [ ]:
confusion_plot(VAL_IDS, pred_lr, "Model E confusion matrix (validation)", "4_4_confusion_model_e")

fig, ax = plt.subplots(figsize=(5.2, 4.4))
for i, c in enumerate(clf.classes_):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], proba[:, i])
    ax.plot(fpr, tpr, lw=2, color=PATH_COLOR[c], label=f"{c} (AUC {roc_auc_score(y_val_bin[:, i], proba[:, i]):.3f})")
ax.plot([0, 1], [0, 1], ls="--", lw=1, color=INK_2)
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("Model E ROC curves, one-vs-rest (validation)", loc="left")
ax.legend(loc="lower right", fontsize=8)
save(fig, "4_4_roc_model_e")

terms = np.array(vectorizer.get_feature_names_out())
top = pd.DataFrame({c: terms[np.argsort(clf.coef_[i])[::-1][:10]] for i, c in enumerate(clf.classes_)})
display(Markdown("**Top 10 terms per class** (largest coefficients; a check that the model reads clinical signal)"))
top

### 4.4.4 Model A: local LLM on the GPU

The pilot runs the first `N_PILOT` validation cases. It is only a check that the setup works, and only the submitted outputs are scored. With `RUN_FULL_VALIDATION = True` the run covers all validation cases as a fixed cohort: a missing or unparseable output counts as a failure, and cases are never retried for a better answer. Plans are appended to `results/model_runs/` as they are generated (gitignored; case IDs and plan fields only), so an interrupted run picks up where it stopped.

In [ ]:
model_a_ready = False
if RUN_MODEL_A:
    try:
        DIGEST = client.digest()
        model_a_ready = True
        print(f"Ollama at {client.host}, model {OLLAMA_MODEL}, digest {DIGEST[:12]}…")
    except Exception as err:
        print(f"Model A skipped: {type(err).__name__}: {err}\n"
              f"Start the server (`ollama serve`) and pull the model (`ollama pull {OLLAMA_MODEL}`), then rerun from here.")
else:
    print("Model A skipped (RUN_MODEL_A = False)")

if model_a_ready:
    run_ids = VAL_IDS if RUN_FULL_VALIDATION else VAL_IDS[:N_PILOT]
    tag = "val" if RUN_FULL_VALIDATION else f"pilot{N_PILOT}"
    run_file = ROOT / "results" / "model_runs" / f"model_a_{OLLAMA_MODEL.replace(':', '_').replace('/', '_')}_{tag}_seed{SEED}.jsonl"
    metadata = {"model": OLLAMA_MODEL, "model_digest": DIGEST, "prompt_sha256": A.prompt_sha256(), "seed": SEED,
                "temperature": TEMPERATURE, "top_p": TOP_P, "max_new_tokens": MAX_NEW_TOKENS}
    run_file.parent.mkdir(parents=True, exist_ok=True)
    run_file.with_suffix(".metadata.json").write_text(json.dumps(metadata, indent=1))
    t0 = time.perf_counter()
    gen_log = A.generate_plans(client, inputs, run_ids, run_file, seed=SEED, temperature=TEMPERATURE, top_p=TOP_P,
                               max_new_tokens=MAX_NEW_TOKENS, num_ctx=NUM_CTX)
    print(f"generated {len(gen_log)} new plans in {time.perf_counter() - t0:.0f} s; file: {run_file.relative_to(ROOT)}")

In [ ]:
if model_a_ready:
    plans_a = [p for p in A.load_plans(run_file) if p["case_id"] in set(run_ids)]
    pred_a = {p["case_id"]: p["diagnosis"] for p in plans_a}
    ver_a, summary_a = verifier_metrics(plans_a, run_ids, fixed_cohort=RUN_FULL_VALIDATION)
    log = gen_log if len(gen_log) else pd.DataFrame(columns=["seconds", "output_tokens", "ok"])
    name_a = f"Model A ({OLLAMA_MODEL})" + ("" if RUN_FULL_VALIDATION else f", pilot {len(run_ids)}")
    RESULTS[name_a] = {**diagnosis_metrics(name_a, run_ids, pred_a), **ver_a,
                       "ms_per_case": log["seconds"].mean() * 1000 if len(log) else None,
                       "output_tokens_per_s": (log["output_tokens"].sum() / log["seconds"].sum()) if len(log) and log["seconds"].sum() else None}
    display(pd.Series(RESULTS[name_a]).to_frame("value"))
    display(Markdown("**Plan status** (verifier)"))
    display(pd.Series(summary_a["status"]).rename_axis("status").reset_index(name="plans"))
    display(Markdown("**Violations by rule**"))
    display(pd.Series(summary_a["violations_by_rule"], dtype=int).rename_axis("rule").reset_index(name="plans"))
    confusion_plot(run_ids, pred_a, f"Model A confusion matrix ({'validation' if RUN_FULL_VALIDATION else 'pilot'})", "4_4_confusion_model_a")

### 4.4.5 Results side by side

In [ ]:
comparison = pd.DataFrame(RESULTS.values())
cols = ["model", "cases", "accuracy", "macro_f1", "macro_precision", "macro_recall", "roc_auc_macro_ovr", "log_loss",
        "supported_checked_acceptance_rate", "joint_success_rate", "n_missing_outputs", "n_unparseable", "ms_per_case"]
comparison = comparison[[c for c in cols if c in comparison.columns]]
display(comparison.round(3))
save_table(comparison.round(4), "4_4_model_comparison")

acc = pd.DataFrame(RESULTS.values()).set_index("model")[[f"acc_{c}" for c in PATH_ORDER]] * 100
fig, ax = plt.subplots(figsize=(8.5, 3.8))
width = 0.8 / len(acc)
model_colors = ["#c9c8c2", "#6250d6", "#008300", "#e87ba4"]       # models get their own hues, distinct from pathologies
for i, (m, row) in enumerate(acc.iterrows()):
    bars = ax.bar(np.arange(4) + (i - (len(acc) - 1) / 2) * width, row.values, width * 0.92, color=model_colors[i % 4], label=m)
    ax.bar_label(bars, labels=[f"{v:.0f}" for v in row.values], fontsize=7.5, color=INK_2, padding=2)
ax.set_xticks(range(4), PATH_ORDER); ax.set_ylim(0, 110); ax.set_ylabel("diagnosis accuracy (%)")
ax.set_title("Diagnosis accuracy by pathology (validation)", loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=min(3, len(acc)), fontsize=8.5)
ax.grid(axis="x", visible=False)
save(fig, "4_4_accuracy_by_pathology")

**Reading the numbers.**
- Diagnosis accuracy here comes from the *full-information* setting: the whole record is given at once. Hager et al.'s 54.9% (OASST 70B) is the *autonomous* setting, where the model has to request each test. The two numbers are not comparable.
- Supported checked acceptance has no numeric target (protocol v1.1). Even a perfect diagnosis cannot push it above roughly 23 to 25% on the covered cases, because most record facts are unknown (`docs/evaluation_feasibility.md`). A low value is therefore mostly a coverage limit of the record, not a model failure.
- Escape rate and the 0.85 adjudicated validity need independent human plan labels. They are reported as unavailable, never estimated.

## 4.5 Data pipeline demo: source → warehouse → monitoring

The same pipeline functions from section 3 are packaged as `src/clinverify/warehouse.py` and load a local **DuckDB** warehouse at `MIMIC_DIR/warehouse/clinverify.duckdb`. DuckDB is a single-file analytical database: it needs no server, and the data stays on the authorised machine as the data-use agreement requires.

| Layer | Contents | Written by |
|---|---|---|
| `raw` | the downloaded tables as read (Ext-CDM, lab mapping, pathology labels, MIMIC-IV admission ↔ patient map) | ingest |
| `staging` | cleaned + transformed tables (deduplicated, normalised text, parsed and flagged labs) | stage |
| `mart` | `cases` (one row per case, with split and patient), `case_facts`, `model_inputs` | mart |
| `ops` | `pipeline_runs`, `step_log`, `dq_results`: counts only, no patient text | every run |

**Data-quality gate.** The raw checks run in memory before anything is written. If one fails, nothing is published and the last good warehouse stays as it was. The run is still logged in `ops`.

In [ ]:
flow_diagram({
    "f":   (0.0, 1.2, 2.0, 1.0, "Source files", "CSV / CSV.GZ", DATA),
    "g":   (2.5, 1.2, 2.0, 1.0, "DQ gate", "validate in memory", "#ffffff"),
    "r":   (5.0, 1.2, 2.0, 1.0, "raw", "as read", STORE),
    "s":   (7.5, 1.2, 2.0, 1.0, "staging", "clean + transform", STORE),
    "m":   (10.0, 1.2, 2.0, 1.0, "mart", "cases, facts, inputs", STORE),
    "c":   (10.0, -0.6, 2.0, 1.0, "DQ checks", "12 SQL checks", "#ffffff"),
    "o":   (5.0, -0.6, 2.0, 1.0, "ops", "runs, steps, checks", STORE),
    "mon": (2.5, -0.6, 2.0, 1.0, "Monitoring", "history, alerts", PROC),
    "cron":(0.0, -0.6, 2.0, 1.0, "Scheduler", "cron / Task Sch.", PROC),
}, [("f", "g", ""), ("g", "r", "pass"), ("r", "s", ""), ("s", "m", ""), ("m", "c", ""), ("c", "o", ""), ("g", "o", "fail: log only"),
    ("o", "mon", ""), ("cron", "f", "trigger")], (11, 3.4), "4_5_warehouse_lineage", "Warehouse lineage, quality gate and monitoring")

### 4.5.1 Run the pipeline (ingest → stage → mart → checks)

In [ ]:
run1 = W.run_pipeline(WAREHOUSE_DB, split_file=SPLIT_FILE, trigger="notebook")
checks1 = run1.pop("checks")
display(pd.Series(run1).to_frame("run"))
save_table(checks1.drop(columns="run_id"), "4_5_dq_checks")

### 4.5.2 What the warehouse holds

In [ ]:
con = W.connect(WAREHOUSE_DB)
layers = W.layer_counts(con)
save_table(layers, "4_5_layer_counts")

In [ ]:
steps = con.execute("SELECT step, table_name, rows_in, rows_out, CASE WHEN step = 'stage' THEN rows_in - rows_out END AS removed, "
                    "n_columns, round(seconds, 3) AS seconds "
                    "FROM ops.step_log WHERE run_id = ? ORDER BY step, table_name", [run1["run_id"]]).df()
display(Markdown("**Row reconciliation per step** (raw → staging: duplicates and empty rows removed)"))
steps

In [ ]:
display(Markdown("**Analytics query on the mart**: cases per split and pathology, with patients and mean input size"))
con.execute('''
    SELECT c.split, c.pathology, COUNT(*) AS cases, COUNT(DISTINCT c.subject_id) AS patients,
           ROUND(AVG(i.input_chars)) AS mean_input_chars, ROUND(AVG(c.n_laboratory_tests), 1) AS mean_labs
    FROM mart.cases c JOIN mart.model_inputs i USING (hadm_id)
    GROUP BY 1, 2 ORDER BY 1, 2''').df()

In [ ]:
display(Markdown("**Sample rows from each layer** (text cut; hidden when SHOW_TEXT_SAMPLES is False)"))
for q in ['SELECT * FROM raw.laboratory_tests LIMIT 3', 'SELECT * FROM staging.laboratory_tests LIMIT 3',
          'SELECT * FROM mart.cases LIMIT 3', 'SELECT hadm_id, pathology, split, input_chars, model_input FROM mart.model_inputs LIMIT 3',
          'SELECT * FROM mart.case_facts LIMIT 3']:
    display(Markdown(f"`{q}`"))
    display(sample(con.execute(q).df()))
con.close()

### 4.5.3 Monitoring: a broken load is caught and not published

The second run feeds the pipeline the deliberately corrupted copy from 3.3.3. The gate catches it, nothing is written, and the run is logged as `failed_checks`. A third, clean run follows. The run history is what an alert or a dashboard reads.

In [ ]:
run2 = W.run_pipeline(WAREHOUSE_DB, split_file=SPLIT_FILE, trigger="notebook-fault-demo", inject_fault=True)
display(run2.pop("checks").drop(columns="run_id"))
run3 = W.run_pipeline(WAREHOUSE_DB, split_file=SPLIT_FILE, trigger="notebook")
run3.pop("checks")
con = W.connect(WAREHOUSE_DB)
history = W.run_history(con, limit=None)
after = W.layer_counts(con)
con.close()
same = after.loc[after.layer != "ops", ["table", "rows"]].reset_index(drop=True).equals(layers.loc[layers.layer != "ops", ["table", "rows"]].reset_index(drop=True))
print("warehouse unchanged by the failed run:", same)
save_table(history.drop(columns=["error"]), "4_5_run_history")

In [ ]:
status_color = {"ok": "#1baf7a", "failed_checks": "#eda100", "error": "#e34948"}
status_mark = {"ok": "o", "failed_checks": "s", "error": "X"}
h = history.sort_values("started_at").reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for s, g in h.groupby("status"):
    axes[0].scatter(g.index + 1, g["duration_s"], s=60, color=status_color.get(s, INK_2), marker=status_mark.get(s, "o"),
                    edgecolor="white", linewidth=1.5, label=s, zorder=3)
axes[0].plot(h.index + 1, h["duration_s"], color=GRID, lw=1.5, zorder=2)
axes[0].set_xlabel("run number"); axes[0].set_ylabel("duration (s)"); axes[0].set_title("Run duration and status", loc="left")
axes[0].legend(fontsize=8)
from matplotlib.ticker import MaxNLocator
for a in axes:
    a.xaxis.set_major_locator(MaxNLocator(integer=True))
passed = h["n_checks"] - h["n_checks_failed"]
axes[1].bar(h.index + 1, passed, color="#1baf7a", width=0.6, label="checks passed")
axes[1].bar(h.index + 1, h["n_checks_failed"], bottom=passed, color="#e34948", width=0.6, label="checks failed", edgecolor="white", linewidth=2)
axes[1].set_xlabel("run number"); axes[1].set_ylabel("checks"); axes[1].set_title("Data-quality checks per run", loc="left")
axes[1].legend(fontsize=8); axes[1].grid(axis="x", visible=False)
fig.tight_layout()
save(fig, "4_5_monitoring")

### 4.5.4 Scheduling and alerting

`scripts/run_pipeline.py` runs the same function from the command line. Its exit code is 0 when the run is ok, 1 when a check fails and 2 on an error, so any scheduler can alert on a non-zero exit.

**Linux / WSL cron** (`crontab -e`), nightly at 02:00:
```
0 2 * * * cd ~/dev/Final_project/Clinical-Discharge-Treatment-Plan-Verification && MIMIC_DIR=$HOME/data/mimic .venv/bin/python scripts/run_pipeline.py --trigger cron >> $HOME/data/mimic/warehouse/cron.log 2>&1
```
**Windows Task Scheduler** (WSL does not keep cron running by itself): action `wsl.exe`, arguments
`-e bash -lc "cd ~/dev/Final_project/Clinical-Discharge-Treatment-Plan-Verification && .venv/bin/python scripts/run_pipeline.py --trigger task-scheduler"`.

Check the history at any time: `python scripts/run_pipeline.py --history`.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "run_pipeline.py"), "--db", str(WAREHOUSE_DB),
                    "--split-file", str(SPLIT_FILE), "--trigger", "cli-demo"], capture_output=True, text=True, cwd=ROOT,
                   env={**os.environ, "MIMIC_DIR": str(MIMIC_DIR)})
print("exit code:", r.returncode)
print(r.stdout[-1500:])

## Run summary and files for the docx

In [ ]:
summary_rows = [
    ("Cases / patients", f"{len(cases):,} / {subject_of.reindex(cases.hadm_id).nunique():,}"),
    ("Raw rows → pre-processed rows", f"{stage_summary.loc[0, 'rows']:,} → {stage_summary.loc[1, 'rows']:,}"),
    ("Validation (clean / corrupted)", f"{'pass' if not problems_good else 'FAIL'} / {len(problems_bad)} problems caught"),
    ("Splits (train / val / test)", " / ".join(f"{len(v):,}" for v in SPLITS.values())),
    ("Warehouse checks (last clean run)", f"{run3['n_checks'] - run3['n_checks_failed']} of {run3['n_checks']} passed"),
    ("Rule catalog", f"v{CATALOG_VERSION}, {len(catalog)} rules"),
] + [(f"{m}: accuracy / macro F1", f"{r['accuracy']:.3f} / {r['macro_f1']:.3f}") for m, r in RESULTS.items()]
display(pd.DataFrame(summary_rows, columns=["item", "value"]))
print("figures:", ", ".join(sorted(p.name for p in FIG.glob("*.png"))))
print("tables :", ", ".join(sorted(p.name for p in OUT.glob("*.csv"))))